In [ ]:
# @title 📦 Setup (run this cell first)
# --- Google Colab setup ---
import os
if os.path.exists('/content') and not os.path.exists('/content/stable_diffusion_carlos'):
    print('📥 Cloning repository...')
    os.system('git clone https://github.com/eth-bmai-fs26/stable_diffusion_carlos.git /content/stable_diffusion_carlos')
if os.path.exists('/content/stable_diffusion_carlos'):
    os.chdir('/content/stable_diffusion_carlos')
    print(f'📂 Working directory: {os.getcwd()}')

%matplotlib inline
import math
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from IPython.display import display, HTML, clear_output

# Matplotlib style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.facecolor'] = 'white'

# Okabe-Ito palette
COLORS = {
    'text': '#E69F00', 'image': '#0072B2', 'connection': '#56B4E9',
    'failure': '#D55E00', 'crossmodal': '#CC79A7', 'neutral': '#999999'
}

print('✅ Setup complete.')

In [ ]:
# @title 🔧 Helper Functions (collapsed)

SHAPES = ['circle', 'square', 'triangle']
COLORS_MAP = {
    'red': (230, 25, 25),
    'blue': (25, 25, 230),
    'green': (25, 204, 25),
}
SIZES = {'small': 5, 'large': 10}
POSITIONS = {
    'center': (16, 16),
    'offset': [(8, 8), (8, 24), (24, 8), (24, 24)],
}

def generate_shape(shape, color, size='large', position='center', img_size=32):
    img = Image.new('RGB', (img_size, img_size), (0, 0, 0))
    draw = ImageDraw.Draw(img)
    rgb = COLORS_MAP[color]
    radius = SIZES[size]
    if position == 'center':
        cx, cy = 16, 16
    else:
        rng = np.random.RandomState()
        cx, cy = POSITIONS['offset'][rng.randint(len(POSITIONS['offset']))]
    if shape == 'circle':
        draw.ellipse([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'square':
        draw.rectangle([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'triangle':
        pts = [(cx, cy-radius), (cx-radius, cy+radius), (cx+radius, cy+radius)]
        draw.polygon(pts, fill=rgb)
    return np.array(img, dtype=np.float32) / 255.0

def generate_dataset(include_size=False, include_position=False):
    sizes = list(SIZES.keys()) if include_size else ['large']
    positions = ['center', 'offset'] if include_position else ['center']
    dataset = []
    for color in COLORS_MAP:
        for shape in SHAPES:
            for sz in sizes:
                for pos in positions:
                    img = generate_shape(shape, color, size=sz, position=pos)
                    label = f"{sz} {color} {shape}" if include_size else f"{color} {shape}"
                    dataset.append((img, label))
    return dataset

class ShapeDataset(torch.utils.data.Dataset):
    def __init__(self, include_size=False, include_position=False):
        raw = generate_dataset(include_size=include_size, include_position=include_position)
        self.images, self.labels, self.label_texts = [], [], []
        unique_labels = []
        for _, label in raw:
            if label not in unique_labels:
                unique_labels.append(label)
        self.label_to_idx = {l: i for i, l in enumerate(unique_labels)}
        for img, label in raw:
            tensor = torch.from_numpy(img).permute(2, 0, 1)
            self.images.append(tensor)
            self.labels.append(self.label_to_idx[label])
            self.label_texts.append(label)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx], self.label_texts[idx]

# --- Visualization helpers ---
def plot_image_grid(images, titles=None, ncols=3, figsize=None, suptitle=None):
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    if figsize is None:
        figsize = (ncols * 2.5, nrows * 2.5)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, torch.Tensor):
                if img.dim() == 3 and img.shape[0] in (1, 3):
                    img = img.permute(1, 2, 0)
                img = img.detach().cpu().numpy()
            img = np.clip(img, 0, 1)
            ax.imshow(img)
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=12)
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_denoising_filmstrip(images, steps=None, title='Denoising Process'):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 2.5))
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        if isinstance(img, torch.Tensor):
            if img.dim() == 3 and img.shape[0] in (1, 3):
                img = img.permute(1, 2, 0)
            img = img.detach().cpu().numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        label = f't={steps[i]}' if steps else f'Step {i}'
        ax.set_title(label, fontsize=10)
        ax.axis('off')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_guidance_comparison(images, scales, title='Guidance Scale Comparison'):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(n * 2.5, 3))
    if n == 1:
        axes = [axes]
    labels_map = {1.0: 'Weak', 3.0: 'Moderate', 7.5: 'Balanced', 10.0: 'Strong', 15.0: 'Extreme'}
    for ax, img, scale in zip(axes, images, scales):
        if isinstance(img, torch.Tensor):
            if img.dim() == 3 and img.shape[0] in (1, 3):
                img = img.permute(1, 2, 0)
            img = img.detach().cpu().numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        label = labels_map.get(scale, '')
        ax.set_title(f'{label}\n(scale={scale})', fontsize=11)
        ax.axis('off')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_progress_comparison(pre_scores, post_scores, questions):
    """Plot before/after quiz score comparison."""
    n = len(questions)
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(n)
    width = 0.35
    bars_pre = ax.bar(x - width/2, pre_scores, width, label='Pre-Test',
                       color=COLORS['neutral'], alpha=0.7)
    bars_post = ax.bar(x + width/2, post_scores, width, label='Post-Test',
                        color=COLORS['connection'], alpha=0.9)
    for bar, val in zip(bars_pre, pre_scores):
        symbol = '✅' if val == 1 else '❌'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                symbol, ha='center', va='bottom', fontsize=14)
    for bar, val in zip(bars_post, post_scores):
        symbol = '✅' if val == 1 else '❌'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                symbol, ha='center', va='bottom', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Q{i+1}\n{q}' for i, q in enumerate(questions)],
                        fontsize=9, ha='center')
    ax.set_ylabel('Correct', fontsize=12)
    ax.set_ylim(0, 1.5)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Wrong', 'Correct'])
    ax.legend(fontsize=11)
    ax.set_title('Your Learning Journey: Before vs After', fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def color_accuracy(image, expected_color):
    if isinstance(image, torch.Tensor):
        if image.dim() == 3 and image.shape[0] == 3:
            image = image.permute(1, 2, 0)
        image = image.detach().cpu().numpy()
    brightness = image.mean(axis=2)
    mask = brightness > 0.1
    if mask.sum() == 0:
        return False
    avg_color = image[mask].mean(axis=0)
    channel_map = {'red': 0, 'blue': 2, 'green': 1}
    return np.argmax(avg_color) == channel_map[expected_color]

def shape_accuracy(image, expected_shape):
    if isinstance(image, torch.Tensor):
        if image.dim() == 3 and image.shape[0] == 3:
            image = image.permute(1, 2, 0)
        image = image.detach().cpu().numpy()
    brightness = image.mean(axis=2)
    gen_mask = (brightness > 0.15).astype(np.float32)
    if gen_mask.sum() < 5:
        return False
    templates = {}
    for shape in ['circle', 'square', 'triangle']:
        tpl = Image.new('L', (32, 32), 0)
        draw = ImageDraw.Draw(tpl)
        r, cx, cy = 10, 16, 16
        if shape == 'circle':
            draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=255)
        elif shape == 'square':
            draw.rectangle([cx-r, cy-r, cx+r, cy+r], fill=255)
        elif shape == 'triangle':
            draw.polygon([(cx, cy-r), (cx-r, cy+r), (cx+r, cy+r)], fill=255)
        templates[shape] = np.array(tpl, dtype=np.float32) / 255.0
    gen_norm = gen_mask - gen_mask.mean()
    gen_std = gen_norm.std()
    if gen_std < 1e-6:
        return False
    best_shape, best_corr = None, -2.0
    for shape, tpl_mask in templates.items():
        tpl_norm = tpl_mask - tpl_mask.mean()
        tpl_std = tpl_norm.std()
        if tpl_std < 1e-6:
            continue
        corr = (gen_norm * tpl_norm).sum() / (gen_std * tpl_std * gen_mask.size)
        if corr > best_corr:
            best_corr = corr
            best_shape = shape
    return best_shape == expected_shape

def validate_prompt(prompt):
    """Validate prompt against known vocabulary. Returns (known, unknown, cleaned)."""
    vocab = {'red', 'blue', 'green', 'circle', 'square', 'triangle',
             'small', 'large', 'center', 'offset'}
    words = prompt.lower().split()
    known = [w for w in words if w in vocab]
    unknown = [w for w in words if w not in vocab]
    cleaned = ' '.join(known) if known else '<pad>'
    return known, unknown, cleaned

print('✅ Helpers loaded.')

In [ ]:
# @title 🏗️ Models & Pre-trained Weights (collapsed)

VOCAB = [
    'red', 'blue', 'green', 'circle', 'square', 'triangle',
    'small', 'large', 'center', 'offset', '<pad>', '<unk>'
]
WORD_TO_IDX = {w: i for i, w in enumerate(VOCAB)}

def tokenize(text, max_len=4):
    tokens = []
    for word in text.lower().split():
        tokens.append(WORD_TO_IDX.get(word, WORD_TO_IDX['<unk>']))
    if len(tokens) < max_len:
        tokens += [WORD_TO_IDX['<pad>']] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    return torch.tensor(tokens, dtype=torch.long)

class TextEncoder(nn.Module):
    def __init__(self, vocab_size=12, embed_dim=32, hidden_dim=64, out_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=WORD_TO_IDX['<pad>'])
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)
    def forward(self, token_ids):
        x = self.embedding(token_ids)
        mask = (token_ids != WORD_TO_IDX['<pad>']).unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

    def encode_tokens(self, token_ids):
        """Per-token embeddings for cross-attention (before pooling)."""
        x = self.embedding(token_ids)
        mask = (token_ids != WORD_TO_IDX['<pad>']).unsqueeze(-1).float()
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x * mask

class ImageEncoder(nn.Module):
    def __init__(self, input_dim=3072, out_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, out_dim)
    def forward(self, images):
        x = images.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

    def encode_tokens(self, token_ids):
        """Per-token embeddings for cross-attention (before pooling)."""
        x = self.embedding(token_ids)
        mask = (token_ids != WORD_TO_IDX['<pad>']).unsqueeze(-1).float()
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x * mask

class MiniCLIP(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_encoder = TextEncoder()
        self.image_encoder = ImageEncoder()
    def forward(self, images, token_ids):
        return self.text_encoder(token_ids), self.image_encoder(images)
    def compute_loss(self, text_emb, image_emb, tau=0.07):
        logits = torch.matmul(text_emb, image_emb.t()) / tau
        labels = torch.arange(len(logits), device=logits.device)
        return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2

class CrossAttention(nn.Module):
    def __init__(self, img_dim=128, text_dim=32, head_dim=32):
        super().__init__()
        self.W_q = nn.Linear(img_dim, head_dim)
        self.W_k = nn.Linear(text_dim, head_dim)
        self.W_v = nn.Linear(text_dim, head_dim)
        self.W_out = nn.Linear(head_dim, img_dim)
        self.scale = head_dim ** -0.5
    def forward(self, image_features, text_features):
        Q = self.W_q(image_features)
        K = self.W_k(text_features)
        V = self.W_v(text_features)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn, dim=-1)
        out = torch.matmul(attn_weights, V)
        out = self.W_out(out)
        return out, attn_weights

class MicroUNet(nn.Module):
    def __init__(self, time_emb_dim=32, text_emb_dim=32):
        super().__init__()
        self.enc1 = nn.Conv2d(3, 32, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, 32)
        self.enc2 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.norm2 = nn.GroupNorm(8, 64)
        self.enc3 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.norm3 = nn.GroupNorm(8, 128)
        self.time_mlp = nn.Sequential(nn.Linear(time_emb_dim, 128), nn.ReLU())
        self.cross_attn = CrossAttention(img_dim=128, text_dim=text_emb_dim, head_dim=32)
        self.dec1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.norm4 = nn.GroupNorm(8, 64)
        self.dec2 = nn.ConvTranspose2d(128, 32, 4, stride=2, padding=1)
        self.norm5 = nn.GroupNorm(8, 32)
        self.final = nn.Conv2d(64, 3, 3, padding=1)
    def forward(self, x, t_emb, text_emb):
        out, _ = self.forward_with_attention(x, t_emb, text_emb)
        return out
    def forward_with_attention(self, x, t_emb, text_emb):
        e1 = F.relu(self.norm1(self.enc1(x)))
        e2 = F.relu(self.norm2(self.enc2(e1)))
        e3 = F.relu(self.norm3(self.enc3(e2)))
        t = self.time_mlp(t_emb)
        e3 = e3 + t[:, :, None, None]
        B, C, H, W = e3.shape
        patches = e3.permute(0, 2, 3, 1).reshape(B, H * W, C)
        if text_emb.dim() == 2:
            text_feat = text_emb.unsqueeze(1)
        else:
            text_feat = text_emb
        attn_out, attn_weights = self.cross_attn(patches, text_feat)
        e3 = (patches + attn_out).reshape(B, H, W, C).permute(0, 3, 1, 2)
        d1 = F.relu(self.norm4(self.dec1(e3)))
        d1 = torch.cat([d1, e2], dim=1)
        d2 = F.relu(self.norm5(self.dec2(d1)))
        d2 = torch.cat([d2, e1], dim=1)
        out = self.final(d2)
        return out, attn_weights

class NoiseScheduler:
    def __init__(self, T=50, s=0.008):
        self.T = T
        self.s = s
        steps = torch.arange(T + 1, dtype=torch.float64)
        f = torch.cos((steps / T + s) / (1 + s) * math.pi / 2) ** 2
        alpha_bars = f / f[0]
        alpha_bars = alpha_bars.clamp(min=1e-5, max=1.0)
        betas = 1 - alpha_bars[1:] / alpha_bars[:-1]
        betas = betas.clamp(min=0.0001, max=0.999)
        self.betas = betas.float()
        self.alphas = (1 - self.betas).float()
        self.alpha_bars = torch.cumprod(self.alphas, dim=0).float()
    def add_noise(self, x_0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_0)
        alpha_bar_t = self.alpha_bars[t]
        while alpha_bar_t.dim() < x_0.dim():
            alpha_bar_t = alpha_bar_t.unsqueeze(-1)
        return torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * noise
    def sample_step(self, x_t, t, predicted_noise):
        beta_t = self.betas[t]
        alpha_t = self.alphas[t]
        alpha_bar_t = self.alpha_bars[t]
        while beta_t.dim() < x_t.dim():
            beta_t = beta_t.unsqueeze(-1)
            alpha_t = alpha_t.unsqueeze(-1)
            alpha_bar_t = alpha_bar_t.unsqueeze(-1)
        mean = (1 / torch.sqrt(alpha_t)) * (x_t - (beta_t / torch.sqrt(1 - alpha_bar_t)) * predicted_noise)
        if isinstance(t, int) and t == 0:
            return mean
        if isinstance(t, torch.Tensor) and (t == 0).all():
            return mean
        noise = torch.randn_like(x_t)
        return mean + torch.sqrt(beta_t) * noise
    @staticmethod
    def get_time_embedding(t, dim=32):
        if isinstance(t, int):
            t = torch.tensor([t], dtype=torch.float32)
        elif isinstance(t, torch.Tensor) and t.dim() == 0:
            t = t.unsqueeze(0).float()
        else:
            t = t.float()
        half_dim = dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, dtype=torch.float32) * -emb)
        emb = t.unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)

@torch.no_grad()
def generate_image(model, scheduler, text_emb, null_text_emb=None,
                   guidance_scale=7.5, steps=50, seed=42):
    torch.manual_seed(seed)
    x = torch.randn(1, 3, 32, 32)
    intermediates = []
    save_steps = set([steps - 1] + [int(i * (steps - 1) / 7) for i in range(8)])
    for t_idx in reversed(range(steps)):
        t = torch.tensor([t_idx])
        t_emb = scheduler.get_time_embedding(t)
        if null_text_emb is not None and guidance_scale != 1.0:
            noise_cond = model(x, t_emb, text_emb)
            noise_uncond = model(x, t_emb, null_text_emb)
            predicted_noise = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        else:
            predicted_noise = model(x, t_emb, text_emb)
        x = scheduler.sample_step(x, t_idx, predicted_noise)
        if t_idx in save_steps:
            intermediates.append(x[0].clone().clamp(0, 1))
    intermediates.reverse()
    return x[0].clamp(0, 1), intermediates

@torch.no_grad()
def generate_image_with_noise_maps(model, scheduler, text_emb, null_text_emb,
                                    guidance_scale=7.5, steps=50, seed=42, capture_step=25):
    """Generate and capture noise predictions (cond, uncond, guided) at one step."""
    torch.manual_seed(seed)
    x = torch.randn(1, 3, 32, 32)
    captured = None
    for t_idx in reversed(range(steps)):
        t = torch.tensor([t_idx])
        t_emb = scheduler.get_time_embedding(t)
        noise_cond = model(x, t_emb, text_emb)
        noise_uncond = model(x, t_emb, null_text_emb)
        predicted_noise = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        if t_idx == capture_step:
            captured = {
                'uncond': noise_uncond[0].clone(),
                'cond': noise_cond[0].clone(),
                'guided': predicted_noise[0].clone(),
            }
        x = scheduler.sample_step(x, t_idx, predicted_noise)
    return x[0].clamp(0, 1), captured

def train_clip(model, dataset, epochs=100, lr=1e-3, tau=0.07):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    images = torch.stack([dataset[i][0] for i in range(len(dataset))])
    texts = [dataset[i][2] for i in range(len(dataset))]
    token_ids = torch.stack([tokenize(t) for t in texts])
    for epoch in range(epochs):
        optimizer.zero_grad()
        text_emb, image_emb = model(images, token_ids)
        loss = model.compute_loss(text_emb, image_emb, tau=tau)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 20 == 0:
            print(f'CLIP Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')
    return losses

def train_denoiser(model, dataset, scheduler, text_encoder, epochs=300,
                   lr=1e-4, steps_per_epoch=1, cfg_dropout=0.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    images = torch.stack([dataset[i][0] for i in range(len(dataset))])
    texts = [dataset[i][2] for i in range(len(dataset))]
    token_ids = torch.stack([tokenize(t) for t in texts])
    with torch.no_grad():
        text_embs = text_encoder.encode_tokens(token_ids)
        null_tokens = tokenize('<pad> <pad>')
        null_emb = text_encoder.encode_tokens(null_tokens.unsqueeze(0))
    for epoch in range(epochs):
        epoch_loss = 0.0
        for _ in range(steps_per_epoch):
            optimizer.zero_grad()
            B = len(images)
            t = torch.randint(0, scheduler.T, (B,))
            noise = torch.randn_like(images)
            x_t = scheduler.add_noise(images, t, noise)
            t_emb = scheduler.get_time_embedding(t)
            if cfg_dropout > 0:
                drop_mask = torch.rand(B) < cfg_dropout
                cond_embs = text_embs.clone()
                cond_embs[drop_mask] = null_emb
            else:
                cond_embs = text_embs
            predicted_noise = model(x_t, t_emb, cond_embs)
            loss = F.mse_loss(predicted_noise, noise)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / steps_per_epoch)
        if (epoch + 1) % 50 == 0:
            print(f'Denoiser Epoch {epoch+1}/{epochs}, Loss: {losses[-1]:.4f}')
    return losses

# --- Load all models ---
dataset = ShapeDataset()
scheduler = NoiseScheduler(T=50)
clip_model = MiniCLIP()
unet = MicroUNet()

weights_loaded = False
for path in ['weights/full_pipeline.pt', '../weights/full_pipeline.pt',
             '/content/weights/full_pipeline.pt']:
    if os.path.exists(path):
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
        clip_model.text_encoder.load_state_dict(
            {k: v.float() for k, v in checkpoint['clip_text'].items()})
        clip_model.image_encoder.load_state_dict(
            {k: v.float() for k, v in checkpoint['clip_image'].items()})
        unet.load_state_dict(
            {k: v.float() for k, v in checkpoint['unet'].items()})
        weights_loaded = True
        print(f'✅ Loaded pre-trained weights from {path}')
        break

if not weights_loaded:
    print('⚠️ Pre-trained weights not found. Training from scratch (~8 min)...')
    train_clip(clip_model, dataset, epochs=100)
    clip_model.eval()
    train_denoiser(unet, dataset, scheduler, clip_model.text_encoder,
                   epochs=1500, lr=3e-4, steps_per_epoch=10, cfg_dropout=0.15)
    print('✅ Training complete.')

clip_model.eval()
unet.eval()

# Pre-compute null embedding for CFG
null_tokens = tokenize('<pad> <pad>')
null_text_emb = clip_model.text_encoder.encode_tokens(null_tokens.unsqueeze(0))

# Pre-compute text embeddings for all 9 base prompts
base_prompts = [dataset[i][2] for i in range(len(dataset))]
all_text_embs = {}
for prompt in base_prompts:
    tokens = tokenize(prompt)
    all_text_embs[prompt] = clip_model.text_encoder.encode_tokens(tokens.unsqueeze(0))

print('✅ All models ready.')

In [ ]:
# Runtime check
try:
    _ = len(COLORS)
    _ = unet
    print('✅ Ready to go!')
except NameError:
    print('⚠️ Please run the setup cells above (grey cells 1-3).')
    print('   Click the ▶ button on each grey cell at the top.')

# 🚗 Notebook 5: The Road Trip — The Full Pipeline

You've built every piece of the car separately. Now we start the engine and drive.

In Notebooks 1–4, you learned:
- **NB1:** Images and text are just numbers (tensors and embeddings)
- **NB2:** CLIP creates a shared language between text and images
- **NB3:** The diffusion engine removes noise step by step
- **NB4:** Cross-attention lets the text guide what the model generates

Now we wire them together, type a prompt, and watch an image appear from pure noise.

> **🧭 Orient:** *"Start the engine, set the destination, drive."*

In [ ]:
# DEMO: The Full Pipeline in Action
# Generate all 9 base classes and see how well the model does.

print('Generating all 9 classes...')
gen_images = []
gen_results = []
for prompt in base_prompts:
    text_emb = all_text_embs[prompt]
    img, _ = generate_image(unet, scheduler, text_emb, null_text_emb,
                             guidance_scale=7.5, seed=42)
    gen_images.append(img)
    parts = prompt.split()
    c_ok = color_accuracy(img, parts[0])
    s_ok = shape_accuracy(img, parts[1])
    gen_results.append((c_ok, s_ok))

# Display 3x3 grid with report card
titles = []
for i, prompt in enumerate(base_prompts):
    c_ok, s_ok = gen_results[i]
    c_sym = '✅' if c_ok else '❌'
    s_sym = '✅' if s_ok else '❌'
    titles.append(f'"{prompt}"\nColor {c_sym}  Shape {s_sym}')

fig = plot_image_grid(gen_images, titles=titles, ncols=3,
                       figsize=(9, 9), suptitle='Your Model: All 9 Classes')
plt.show()

# Summary
color_correct = sum(1 for c, s in gen_results if c)
shape_correct = sum(1 for c, s in gen_results if s)
full_correct = sum(1 for c, s in gen_results if c and s)
print(f'\nReport Card:')
print(f'  Color accuracy: {color_correct}/9 ({color_correct/9:.0%})')
print(f'  Shape accuracy: {shape_correct}/9 ({shape_correct/9:.0%})')
print(f'  Full match:     {full_correct}/9 ({full_correct/9:.0%})')
print(f'\nYour model gets {full_correct}/9 fully correct. The failures are interesting \u2014')
print('they reveal the limits of a tiny model, not a broken one.')

---

Every one of those images started as **pure random noise** and was refined into a recognizable shape in 50 denoising steps, guided only by a 2-word text prompt.

Let's explore how the pipeline works in detail.

---

In [ ]:
# PROMPT LAB: Explore the pipeline interactively
# Pick a prompt, a seed, and a guidance scale. Watch it generate.

# Extended prompts: base 9 + 9 with size
extended_prompts = list(base_prompts)
for sz in ['small', 'large']:
    for color in ['red', 'blue', 'green']:
        for shape in ['circle', 'square', 'triangle']:
            ep = f'{sz} {color} {shape}'
            if ep not in extended_prompts:
                extended_prompts.append(ep)

@interact(
    prompt=Dropdown(options=extended_prompts, value=extended_prompts[0], description='Prompt:'),
    seed=IntSlider(min=0, max=999, value=42, description='Seed:'),
    guidance_scale=FloatSlider(min=1.0, max=15.0, step=0.5, value=7.5, description='Guidance:')
)
def prompt_lab(prompt, seed, guidance_scale):
    # Encode prompt
    tokens = tokenize(prompt)
    text_emb = clip_model.text_encoder.encode_tokens(tokens.unsqueeze(0))

    # Generate
    img, frames = generate_image(unet, scheduler, text_emb, null_text_emb,
                                  guidance_scale=guidance_scale, seed=seed)

    # Filmstrip
    fig = plot_denoising_filmstrip(frames,
        title=f'Generating "{prompt}" (seed={seed}, guidance={guidance_scale})')
    plt.show()

    # Mini report card
    parts = prompt.split()
    color_word = [p for p in parts if p in ['red', 'blue', 'green']]
    shape_word = [p for p in parts if p in ['circle', 'square', 'triangle']]
    if color_word and shape_word:
        c_ok = color_accuracy(img, color_word[0])
        s_ok = shape_accuracy(img, shape_word[0])
        c_sym = '✅' if c_ok else '❌'
        s_sym = '✅' if s_ok else '❌'
        print(f'Color: {c_sym}  Shape: {s_sym}')
    print(f'Try different seeds to see how randomness creates variety!')

---

> **Try it:** Change the seed and watch a completely different image emerge from the same prompt. Same destination, different roads. That's stochasticity by design, not a bug.

---

### The Complete Generation Function

Here is the **entire generation pipeline** in one function. This is the most important code in the course. Every component you built in Notebooks 1–4 is present:

In [ ]:
# THE COMPLETE PIPELINE -- every component in one function
def generate(prompt, seed=42, guidance_scale=7.5):
    # Step 1: CLIP encodes the text prompt (NB2)
    tokens = tokenize(prompt)
    text_emb = clip_model.text_encoder.encode_tokens(tokens.unsqueeze(0))
    null_emb = null_text_emb  # empty prompt for CFG

    # Step 2: Start from pure noise (NB3)
    torch.manual_seed(seed)
    x = torch.randn(1, 3, 32, 32)

    # Step 3: Denoise for 50 steps (NB3 + NB4)
    for t in reversed(range(50)):
        t_emb = scheduler.get_time_embedding(t)
        # Two forward passes: with and without text
        cond_noise = unet(x, t_emb, text_emb)       # "What the prompt wants"
        uncond_noise = unet(x, t_emb, null_emb)      # "What noise looks like"
        # THIS ONE LINE is why text controls image generation:
        guided_noise = uncond_noise + guidance_scale * (cond_noise - uncond_noise)
        x = scheduler.sample_step(x, t, guided_noise)

    return x[0].clamp(0, 1)

# Demo
result = generate('red circle', seed=42)
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(result.permute(1, 2, 0).numpy())
ax.set_title('generate("red circle")', fontsize=13)
ax.axis('off')
plt.show()

print('That\'s it. The entire pipeline is:')
print('  1. CLIP encodes your prompt into a 32-dimensional vector')
print('  2. Start from random noise')
print('  3. At each of 50 steps, the model runs TWICE:')
print('     - Once WITH your prompt  (conditional noise prediction)')
print('     - Once WITHOUT any prompt (unconditional noise prediction)')
print('  4. The guided noise = uncond + scale * (cond - uncond)')
print('  5. Remove that guided noise from the image')
print('\nThe guidance scale controls how strongly text influences the result.')

---

### Classifier-Free Guidance: The Mechanism Revealed

In Notebook 4, we showed that guidance scale changes the output — but not *how*. Now we reveal the mechanism.

The model runs **twice** per denoising step:
1. **Conditional pass:** "What noise to remove if we're making a red circle?"
2. **Unconditional pass:** "What noise to remove if we don't care what we make?"

The difference between them is the **text signal**. We amplify that difference:

```
guided_noise = uncond_noise + scale * (cond_noise - uncond_noise)
```

- `scale = 1.0`: Just use the conditional prediction (co-pilot whispers)
- `scale = 7.5`: Amplify the text signal 7.5x (co-pilot speaks clearly)
- `scale = 15.0`: Amplify 15x (co-pilot shouts — may overshoot)

---

In [ ]:
# CFG MECHANISM: See the two noise predictions + guided result

prompt_cfg = 'red circle'
text_emb_cfg = all_text_embs[prompt_cfg]

img_cfg, noise_maps = generate_image_with_noise_maps(
    unet, scheduler, text_emb_cfg, null_text_emb,
    guidance_scale=7.5, seed=42, capture_step=25
)

# Display the three noise predictions as heatmaps
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Show noise magnitude per pixel (mean across channels)
for ax, (key, title, cmap) in zip(axes[:3], [
    ('uncond', 'Unconditional noise\n("What does noise look like?")', 'Greys'),
    ('cond', 'Conditional noise\n("What if it\'s a red circle?")', 'Blues'),
    ('guided', 'Guided noise\n(uncond + 7.5 * difference)', 'Purples'),
]):
    noise_img = noise_maps[key].numpy()
    magnitude = np.abs(noise_img).mean(axis=0)  # Average across channels
    im = ax.imshow(magnitude, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.7)

# Final image
axes[3].imshow(img_cfg.permute(1, 2, 0).numpy())
axes[3].set_title(f'Final result\n"{prompt_cfg}"', fontsize=10)
axes[3].axis('off')

fig.suptitle('Classifier-Free Guidance: Inside One Denoising Step (t=25)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print('The DIFFERENCE between conditional and unconditional is the text signal.')
print('Amplifying that difference = making the model follow text more strongly.')
print('This is why it\'s called "classifier-FREE" guidance: no separate classifier needed.')

---

### STUMBLE: Compositional Complexity

Our model handles 9 basic classes well. But what happens when we add more complexity?

---

In [ ]:
# STUMBLE: Expand to 36 classes (add size + position)
# The binding problem scales with complexity.

complex_prompts = [
    'small red circle', 'large blue square', 'small green triangle',
    'large red square', 'small blue triangle', 'large green circle',
]

complex_images = []
complex_results = []
for prompt in complex_prompts:
    tokens = tokenize(prompt)
    text_emb = clip_model.text_encoder.encode_tokens(tokens.unsqueeze(0))
    img, _ = generate_image(unet, scheduler, text_emb, null_text_emb,
                             guidance_scale=7.5, seed=42)
    complex_images.append(img)
    parts = prompt.split()
    color_word = [p for p in parts if p in ['red', 'blue', 'green']][0]
    shape_word = [p for p in parts if p in ['circle', 'square', 'triangle']][0]
    c_ok = color_accuracy(img, color_word)
    s_ok = shape_accuracy(img, shape_word)
    complex_results.append((c_ok, s_ok))

# Display
titles = []
for i, prompt in enumerate(complex_prompts):
    c_ok, s_ok = complex_results[i]
    c_sym = '✅' if c_ok else '❌'
    s_sym = '✅' if s_ok else '❌'
    titles.append(f'"{prompt}"\nColor {c_sym}  Shape {s_sym}')

fig = plot_image_grid(complex_images, titles=titles, ncols=3,
                       figsize=(9, 7), suptitle='⚠️ Compositional Challenge: 3-Word Prompts')
plt.show()

full_ok = sum(1 for c, s in complex_results if c and s)
print(f'Full success: {full_ok}/{len(complex_prompts)}')
print()
print('When prompts get more complex, the model struggles with "binding":' )
print('it knows what colors and shapes exist, but may attach them incorrectly.')
print('"Small red circle" might become a large red blob or a small blue circle.')
print()
print('This is the same problem that makes real models struggle with')
print('"a red cat sitting on a blue chair" \u2014 it\'s fundamental, not a bug.')

---

> **💡 Aha!** The binding problem scales with prompt complexity. Two attributes (color + shape) work well. Three (size + color + shape) is harder. Real-world prompts with many objects and attributes are harder still. This isn't a flaw in *our* tiny model — it's an active research problem in models 25,000× larger.

---

---

<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0;">

### 💼 Manager's Briefing: The Prompt Engineering Economy

Small wording changes create large output differences. "A red circle" and "circle, red" produce different CLIP embeddings, which change cross-attention steering nonlinearly. **Prompt sensitivity is architectural, not a model flaw.**

This is why "prompt engineering" became a job title. It's not magic — it's understanding how text encodes into embeddings and how those embeddings steer attention.

**Questions to ask:**
1. *"Does your model support negative prompts?"* (Co-pilot saying "NOT that way")
2. *"What guidance scale do you default to?"* (7–10 is standard; higher = more literal)
3. *"How does your model handle multi-attribute prompts?"*

</div>

---

---

<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0;">

### 💼 Real-World Story: The Prompt Engineering Economy

A marketing team needed 50 product mockup variations. Their first attempt used simple prompts ("blue sneaker on white background") and got inconsistent results: wrong angles, missing details, odd shadows. A prompt engineer rewrote them with precise language ("product photography of a blue running sneaker, centered, white studio background, soft shadow, 3/4 angle") and hit rate jumped from 20% to 85%.

**Formal diagnosis:** CLIP encodes syntactic structure into embeddings. "Blue sneaker" and "sneaker, blue, product photo" produce *different* 32-dimensional vectors. Different vectors steer cross-attention differently. The relationship between prompt wording and output is nonlinear.

**As a decision-maker:** Prompt engineering is a real skill with real ROI. Budget for it like you would for copywriting.

</div>

---

### Safety: Where Would Filters Go?

Real text-to-image systems need safety filters. Here's where they fit in the pipeline:

In [ ]:
# SAFETY DEMO: What happens with a "blocked" prompt?

# Simulate a safety filter: replace prompt embedding with null
print('Simulating a blocked prompt...')
print('A safety filter replaces the text embedding with a null (empty) embedding.')
print()

# Generate with null embedding (as if prompt was blocked)
blocked_img, blocked_frames = generate_image(
    unet, scheduler, null_text_emb, null_text_emb,
    guidance_scale=7.5, seed=42
)

# Compare: normal vs blocked
normal_img, _ = generate_image(
    unet, scheduler, all_text_embs['red circle'], null_text_emb,
    guidance_scale=7.5, seed=42
)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(normal_img.permute(1, 2, 0).numpy())
axes[0].set_title('✅ Normal: "red circle"', fontsize=12)
axes[0].axis('off')

axes[1].imshow(np.clip(blocked_img.permute(1, 2, 0).numpy(), 0, 1))
axes[1].set_title('🚫 Blocked: null embedding', fontsize=12)
axes[1].axis('off')

fig.suptitle('Safety Filter: Replacing Prompt with Null Embedding',
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print('With a null embedding, the model has no destination \u2014 it produces incoherent noise.')
print('This is the simplest safety mechanism: block at the text encoding stage.')

---

**Three places to put safety filters in the pipeline:**

```
[User Prompt] --filter 1--> [CLIP Text Encoder] --> [Denoising Loop] --filter 2--> [Output Image] --filter 3--> [User]
```

| Filter | What It Does | What It Catches | What It Misses |
|--------|-------------|----------------|----------------|
| **Filter 1: Input text** | Block/modify harmful prompts before encoding | Explicit harmful text | Adversarial rephrasing, implicit harm |
| **Filter 2: Output image** | Classify generated image for harmful content | Explicit harmful imagery | Subtle or context-dependent harm |
| **Filter 3: Post-processing** | Human review, watermarking, metadata | Everything the reviewer catches | Scale limitations |

No single filter is sufficient. Production systems use all three.

---

---

### Bridge to Real Tools

You've built a working text-to-image system with **~300,000 parameters** generating **32×32 images**. Here's how it maps to the real thing:

| What You Built | What The Pros Use | Scale Factor |
|---------------|-------------------|-------------|
| MiniCLIP (~200K params) | OpenCLIP ViT-H/14 (~632M params) | 3,000× |
| MicroUNet (~300K params) | Stable Diffusion UNet (~860M params) | 2,867× |
| 12-word vocabulary | 49,408-token BPE vocabulary | 4,100× |
| 32×32 pixels | 512×512 to 1024×1024 pixels | 256–1,024× |
| 9 shape classes | Billions of internet images | ~100M× |
| 50 denoising steps | 20–50 steps (with better schedulers) | Same |
| Cross-attention at bottleneck | Cross-attention at multiple resolutions | 4–8× |

**The architecture is the same.** You understand every component.

---

### Decision Tree: API vs Fine-Tune vs Self-Host

Now that you understand the architecture, here's how to choose:

```
Do you need custom image generation?
│
├─ NO → Use stock images or templates
│
└─ YES → How many images per month?
         │
         ├─ < 1,000 → Use an API (DALL-E, Midjourney, Stability AI)
         │           Key question: cost per image, rate limits
         │
         ├─ 1,000–50,000 → Do you need a custom style?
         │    │
         │    ├─ NO → API is still cheapest
         │    └─ YES → Fine-tune (LoRA/DreamBooth, 20–100 examples)
         │                Key question: how many style examples needed
         │
         └─ > 50,000 → Self-host open source (SDXL, Flux)
                      Key question: GPU infrastructure cost
                      Minimum: 1× A10G per concurrent request
```

> **When in doubt:** Start with an API, measure demand, then decide.

---

### Jargon Decoder

When you hear these terms, you now know what they mean:

| Industry Term | What You Learned | Course Concept |
|--------------|-----------------|----------------|
| **Latent space** | The 32-dim shared embedding space | Where CLIP places text and images (NB2) |
| **LoRA** | Low-rank fine-tuning of the UNet | Adjusting ~1% of denoiser weights for a custom style |
| **CFG scale** | Guidance scale | How strongly text steers generation (NB5) |
| **Negative prompt** | A second text embedding subtracted from guidance | Co-pilot saying "NOT that way" |
| **Sampling steps** | Denoising steps (we used 50) | Each step = 1–2 full model passes (NB3) |
| **Checkpoint** | Saved model weights (our full_pipeline.pt) | Snapshot of all learned parameters |
| **img2img** | Start denoising from a noisy *real* image instead of pure noise | Same pipeline, different starting point |
| **Inpainting** | Denoise only selected regions, keep the rest | Masked denoising loop |
| **Tokenizer** | Our `tokenize()` function | Converts words to numbers for the text encoder |
| **Embedding** | Output of CLIP's text encoder (32-dim vector) | The "map" the co-pilot reads (NB2) |

### 🚩 Vendor Red Flags

If a vendor says any of these, ask follow-up questions:

1. 🚩 **"Our model has no bias."** — CLIP learns biases from training data. All models have bias. Ask: *"How do you audit for bias?"*

2. 🚩 **"We guarantee no harmful content."** — No filter catches everything. Ask: *"What's your false negative rate on safety filters?"*

3. 🚩 **"Our model perfectly understands prompts."** — Cross-attention has fundamental binding limits. Ask: *"Show me your compositional accuracy benchmark."*

4. 🚩 **"Higher resolution at no extra cost."** — Doubling resolution quadruples compute. Ask: *"What's the native generation resolution vs upscaled?"*

5. 🚩 **"We trained our model from scratch."** — Training Stable Diffusion from scratch costs $600K+. Most fine-tune existing models. Ask: *"What base model did you start from?"*

---

### 📋 Complete Co-Pilot Reference Card (All 5 Notebooks)

| Notebook | Component | What It Does | Co-Pilot Metaphor |
|----------|-----------|-------------|-------------------|
| NB1 | Raw Ingredients | Images = tensors of numbers; text = embedding vectors; cosine sim measures closeness | "Here are the parts of the car and the map" |
| NB2 | CLIP (Translator) | Learns shared embedding space; contrastive loss pulls matching pairs together | "The co-pilot learns to read the map" |
| NB3 | Diffusion (Engine) | Forward: add noise. Reverse: predict & remove noise. UNet preserves spatial structure. | "The driver learns to navigate by being dropped at random locations" |
| NB4 | Cross-Attention (Conversation) | Q=image, K/V=text. Each image region checks the text. Guidance scale = co-pilot volume. | "At every turn, the driver asks the co-pilot" |
| **NB5** | **Full Pipeline (Road Trip)** | **Wire it all together: noise → denoise with text guidance → image. CFG = two passes.** | **"Start the engine, set the destination, drive"** |

### 5 Questions to Ask a Vendor

1. **From NB1:** *"What resolution does your model generate natively?"* (Going from 512×512 to 1024×1024 quadruples cost.)
2. **From NB2:** *"How many text-image pairs was your CLIP trained on?"* (Dozens = overfit. Thousands = maybe.)
3. **From NB3:** *"How many denoising steps? Do you support distilled models?"* (Each step = a full forward pass.)
4. **From NB4:** *"What's your compositional accuracy on multi-attribute prompts?"*
5. **From NB5:** *"What's your default guidance scale, and can users adjust it?"*

---

---

### 🎓 Post-Test Quiz

Let's see how much you've learned. These 5 questions test the same concepts as the pre-test, but now applied to real-world scenarios.

---

In [ ]:
# @title 📝 Post-Test Quiz (collapsed)

post_test_questions = [
    {
        'id': 'Q1', 'area': 'CLIP',
        'question': 'Your team says they "fine-tuned CLIP for your product catalog." What did they actually do?',
        'options': [
            'a) Trained a model to generate product images from descriptions',
            'b) Trained a model to place product photos and descriptions near each other in a shared embedding space',
            'c) Built a search engine using keyword matching',
            'd) Compressed the product database for faster retrieval',
        ],
        'answer': 'b'
    },
    {
        'id': 'Q2', 'area': 'Stochasticity',
        'question': 'A client complains: "the AI gives inconsistent results." How do you explain this?',
        'options': [
            'a) The model needs to be retrained with more data',
            'b) This is by design \u2014 different random seeds produce different valid interpretations',
            'c) There is a bug in the deployment pipeline',
            'd) The model is too small and needs more parameters',
        ],
        'answer': 'b'
    },
    {
        'id': 'Q3', 'area': 'Cross-attention',
        'question': 'Your vendor\'s model generates "a silver watch on a wooden table" but the watch looks wooden. What went wrong?',
        'options': [
            'a) The model doesn\'t understand the word "silver"',
            'b) The training data didn\'t include enough watches',
            'c) Cross-attention is bleeding \u2014 the material attributes are applied to the wrong objects',
            'd) The image resolution is too low to show metallic textures',
        ],
        'answer': 'c'
    },
    {
        'id': 'Q4', 'area': 'Failure diagnosis',
        'question': 'Your team\'s custom model produces colorful but blobby images. What architectural change would you suggest?',
        'options': [
            'a) Increase the text vocabulary size',
            'b) Replace the fully-connected denoiser with a convolutional one (UNet) for spatial awareness',
            'c) Use a longer text prompt',
            'd) Increase the number of denoising steps',
        ],
        'answer': 'b'
    },
    {
        'id': 'Q5', 'area': 'Cost',
        'question': 'Your company wants to cut image generation costs by 50%. Which approach would be most effective?',
        'options': [
            'a) Use shorter text prompts',
            'b) Reduce the number of denoising steps or use a distilled model',
            'c) Generate smaller images and upscale them afterward',
            'd) Switch to a different programming language',
        ],
        'answer': 'b'
    },
]

POST_TEST_ANSWERS = {}

for q in post_test_questions:
    print(f"\n{q['id']} ({q['area']}): {q['question']}")
    for opt in q['options']:
        print(f"  {opt}")

print('\n' + '='*60)
print('Use the widget below to record your answers.')

# Interactive quiz
answer_widgets = {}
for q in post_test_questions:
    w = widgets.RadioButtons(
        options=['a', 'b', 'c', 'd'],
        description=f"{q['id']}:",
        disabled=False
    )
    answer_widgets[q['id']] = w
    display(widgets.Label(value=f"{q['id']}: {q['question']}"))
    display(w)

def submit_post_test(btn):
    clear_output(wait=True)
    print('\n\u2705 Post-Test Results:\n')
    correct = 0
    for q in post_test_questions:
        user_ans = answer_widgets[q['id']].value
        POST_TEST_ANSWERS[q['id']] = user_ans
        is_correct = user_ans == q['answer']
        if is_correct:
            correct += 1
        sym = '✅' if is_correct else '❌'
        print(f"  {q['id']} ({q['area']}): {sym} (you: {user_ans}, correct: {q['answer']})")
    print(f'\nScore: {correct}/{len(post_test_questions)}')

submit_btn = widgets.Button(description='Submit Answers', button_style='success')
submit_btn.on_click(submit_post_test)
display(submit_btn)

In [ ]:
# @title 📊 Before vs After Comparison (collapsed)
# (If you didn't take the pre-test in NB1, this uses defaults.)

# Pre-test answers (from NB1 or defaults)
pre_test_correct = {
    'Q1': 'b', 'Q2': 'b', 'Q3': 'c', 'Q4': 'b', 'Q5': 'b'
}

try:
    pre_answers = PRE_TEST_ANSWERS  # Set in NB1 if available
except NameError:
    pre_answers = {'Q1': 'd', 'Q2': 'd', 'Q3': 'd', 'Q4': 'd', 'Q5': 'd'}  # Default: "unsure"

# Get post-test answers
post_answers = POST_TEST_ANSWERS if POST_TEST_ANSWERS else {
    'Q1': 'b', 'Q2': 'b', 'Q3': 'c', 'Q4': 'b', 'Q5': 'b'  # Assume correct if not taken
}

# Compute scores
knowledge_areas = ['CLIP', 'Stochasticity', 'Cross-Attn', 'Diagnosis', 'Cost']
pre_scores = []
post_scores = []
for i, qid in enumerate(['Q1', 'Q2', 'Q3', 'Q4', 'Q5']):
    pre_scores.append(1 if pre_answers.get(qid) == pre_test_correct[qid] else 0)
    post_scores.append(1 if post_answers.get(qid) == pre_test_correct[qid] else 0)

fig = plot_progress_comparison(pre_scores, post_scores, knowledge_areas)
plt.show()

pre_total = sum(pre_scores)
post_total = sum(post_scores)
print(f'Pre-test:  {pre_total}/5')
print(f'Post-test: {post_total}/5')
if post_total > pre_total:
    print(f'\n\U0001f389 You improved by {post_total - pre_total} point(s)!')
elif post_total == pre_total == 5:
    print('\n\U0001f31f Perfect score both times! You came in strong.')
else:
    print('\nEvery question you got right represents a concept you now understand mechanically,')
    print('not just vaguely. That\'s the real value.')

---

<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0;">

### 💼 Final Manager's Briefing

You've now built every component of a text-to-image system at miniature scale. The real Stable Diffusion model is **25,000× larger** but **architecturally identical**.

You know what CLIP, U-Net, cross-attention, guidance scale, and denoising steps mean — **mechanically, not vaguely.** When your engineering team says "we need to increase the guidance scale" or "the model has a binding problem," you know exactly what they're talking about.

That's the difference between a manager who nods along and one who asks the right questions.

</div>

---

In [ ]:
# @title 📄 Export Reference Card (collapsed)

export_content = """============================================================
THE CO-PILOT REFERENCE CARD
Stable Diffusion from Scratch: A Decision-Maker's Guide
============================================================

COMPONENT SUMMARY
-----------------------------------------------------------
NB1 | Raw Ingredients
    | Images = tensors of numbers; text = embedding vectors;
    | cosine sim measures closeness.
    | Metaphor: "Here are the parts of the car and the map"
-----------------------------------------------------------
NB2 | CLIP (Translator)
    | Learns shared embedding space; contrastive loss pulls
    | matching pairs together.
    | Metaphor: "The co-pilot learns to read the map"
-----------------------------------------------------------
NB3 | Diffusion (Engine)
    | Forward: add noise. Reverse: predict & remove noise.
    | UNet preserves spatial structure.
    | Metaphor: "The driver learns to navigate by being
    |            dropped at random locations"
-----------------------------------------------------------
NB4 | Cross-Attention (Conversation)
    | Q=image, K/V=text. Each image region checks the text.
    | Guidance scale = co-pilot volume.
    | Metaphor: "At every turn, the driver asks the co-pilot"
-----------------------------------------------------------
NB5 | Full Pipeline (Road Trip)
    | Wire it all together: noise -> denoise with text
    | guidance -> image. CFG = two passes.
    | Metaphor: "Start the engine, set the destination, drive"
-----------------------------------------------------------

5 QUESTIONS TO ASK A VENDOR
1. What resolution does your model generate natively?
2. How many text-image pairs was your CLIP trained on?
3. How many denoising steps? Do you support distilled models?
4. What's your compositional accuracy on multi-attribute prompts?
5. What's your default guidance scale, and can users adjust it?

JARGON DECODER
- Latent space: Where CLIP places text and images (32-dim vectors)
- LoRA: Low-rank fine-tuning (~1% of denoiser weights)
- CFG scale: How strongly text steers generation
- Negative prompt: Co-pilot saying "NOT that way"
- Sampling steps: Each step = 1-2 full model passes
- Checkpoint: Saved model weights (snapshot of all parameters)
- img2img: Start from a noisy real image instead of pure noise
- Inpainting: Denoise only selected regions
- Tokenizer: Converts words to numbers for the text encoder
- Embedding: 32-dim vector output of CLIP's text encoder

VENDOR RED FLAGS
1. "Our model has no bias." -> Ask: How do you audit?
2. "We guarantee no harmful content." -> Ask: False negative rate?
3. "Our model perfectly understands prompts." -> Ask: Compositional benchmark?
4. "Higher resolution at no extra cost." -> Ask: Native vs upscaled resolution?
5. "We trained from scratch." -> Ask: What base model did you start from?

DECISION TREE
- < 1K images/month -> API (DALL-E, Midjourney, Stability AI)
- 1K-50K + custom style -> Fine-tune (LoRA/DreamBooth)
- > 50K images/month -> Self-host (SDXL, Flux)
- Unsure -> Start with API, measure demand, then decide
"""

# Add quiz scores if available
try:
    pre_total_export = sum(pre_scores)
    post_total_export = sum(post_scores)
    export_content += f"""\nQUIZ SCORES
- Pre-test:  {pre_total_export}/5
- Post-test: {post_total_export}/5
"""
except NameError:
    pass

export_content += """\n============================================================
Generated by: Stable Diffusion from Scratch course
============================================================
"""

# Write to file
export_path = 'stable_diffusion_reference_card.txt'
try:
    # Try Colab path first
    export_path = '/content/stable_diffusion_reference_card.txt'
    with open(export_path, 'w') as f:
        f.write(export_content)
    from google.colab import files
    files.download(export_path)
    print(f'✅ Reference card downloaded!')
except Exception:
    # Fallback: write locally and print
    export_path = 'stable_diffusion_reference_card.txt'
    with open(export_path, 'w') as f:
        f.write(export_content)
    print(f'✅ Reference card saved to: {export_path}')
    print()
    print(export_content)

---

## 🎯 Course Complete

You started with the question: *"How does text become an image?"*

Now you can answer it mechanically:

1. **CLIP** encodes your text into a 32-dimensional vector
2. **The denoiser** starts from pure random noise
3. At each of **50 steps**, cross-attention lets every image region check the text
4. **Classifier-free guidance** amplifies the text signal (two passes per step)
5. After 50 steps, an image appears that matches your prompt

Every component you built individually is now working in concert. The co-pilot reads the map, the driver navigates, and together they reach the destination.

> *You didn't just learn about Stable Diffusion. You built one.*

---